# LTX-2.3 10Eros (Vantage workflow) — Lightning AI Clean ComfyUI Template

## Prerequisites — read this first

This notebook assumes you launched your Lightning AI Studio from the
**Clean ComfyUI Template** (or similar pre-built ComfyUI template):

> https://lightning.ai/mindthemath/studios/clean-comfyui-template-v0-3-15-20250221

That template ships ComfyUI + PyTorch + CUDA already installed and
configured. We don't reinstall those. We only add the Vantage-specific
deltas:

- 11 custom node packs (10S-Comfy-nodes, rgthree-comfy, ComfyUI-GGUF,
  ComfyUI-LTXVideo, KJNodes, RES4LYF, comfy_mtb, etc.)
- 8 Vantage / 10Eros model files (~22 GB)
- mediapipe pin (10S-Comfy-nodes uses removed `mp.solutions` API)
- The Vantage workflow JSON
- A Gradio UI that talks to ComfyUI via its `/prompt` HTTP endpoint

**Launch steps:**

1. Lightning AI dashboard → **New Studio** → search "Clean ComfyUI Template" → Create
2. Switch hardware to GPU if it spawned on CPU (top panel)
3. Upload this notebook (drag onto file explorer or use Lightning's upload)
4. Open the notebook, run cells top-to-bottom
5. After Cell 3 launches Gradio, use the **Custom Port plugin** on the
   right-hand panel to forward port `7860` (Gradio) — that gives you the
   public URL. Port 8188 (ComfyUI) is internal.

**Storage budget:** ~22 GB of model downloads on top of the ~3 GB the
template already provisioned.

**Time budget:** ~10-15 min on first run (mostly the 14 GB 10Eros UNET
download). Re-runs: cached.

In [ ]:
# Cell 1/3 — Detect Lightning's ComfyUI, install ONLY the deltas
# (custom nodes + Vantage models + mediapipe pin + workflow JSON).
# PyTorch + CUDA + base ComfyUI already shipped by the Clean ComfyUI
# Template, so we don't reinstall those.
import os, sys, subprocess, shutil, glob
from pathlib import Path
from IPython.display import display, HTML

def run(cmd, cwd=None, check=True):
    display(HTML(f"<pre style='color:#9ad;font-size:12px'>$ {' '.join(str(c) for c in cmd)}</pre>"))
    return subprocess.run(cmd, cwd=cwd, check=check)

# ---- Auto-detect where the template put ComfyUI. Lightning's various
# templates put it in different places; we try the common ones.
CANDIDATES = [
    "/teamspace/studios/this_studio/ComfyUI",
    "/teamspace/studios/this_studio/comfyui",
    "/workspace/ComfyUI",
    "/root/ComfyUI",
    str(Path.home() / "ComfyUI"),
]
COMFY_PATH = None
for c in CANDIDATES:
    if os.path.exists(os.path.join(c, "main.py")):
        COMFY_PATH = c; break
if COMFY_PATH is None:
    for root in ("/teamspace/studios/this_studio", "/workspace", "/root", str(Path.home())):
        if os.path.exists(root):
            for hit in glob.glob(f"{root}/**/main.py", recursive=True)[:10]:
                if "ComfyUI" in hit or "comfyui" in hit:
                    COMFY_PATH = os.path.dirname(hit); break
        if COMFY_PATH: break
if COMFY_PATH is None:
    BASE = Path("/teamspace/studios/this_studio")
    BASE = BASE if BASE.exists() else Path.cwd()
    COMFY_PATH = str(BASE / "ComfyUI")
    if not os.path.exists(COMFY_PATH):
        display(HTML("<p style='color:#ef6c00'><b>No template ComfyUI found — cloning from scratch.</b></p>"))
        run(["git", "clone", "-q", "https://github.com/comfyanonymous/ComfyUI", COMFY_PATH])
        run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{COMFY_PATH}/requirements.txt"])

BASE_PATH = Path(COMFY_PATH).parent
WF_UI_PATH = str(BASE_PATH / "Vantage-10Eros_I2V_v3.2.json")

# ---- The exact Gemma 3 12B text encoder file the Vantage workflow expects.
# inflatebot/LTX23-gemma-3-12b-it-orthogonal-reflection-bounded-ablation-v4-fp4_mixed
# is the CANONICAL community variant: 9.45 GB single-file safetensors, FP4
# mixed precision, ORTHOGONAL REFLECTION BOUNDED ABLATION v4 (a soft form of
# abliteration — Google's safety dampening is removed without hurting general
# quality). Drop-in compatible with Vantage's DualCLIPLoader slot 1.
# We DON'T use the GGUF heretic variant because ComfyUI-GGUF's
# DualCLIPLoaderGGUF doesn't reliably mix with the LTX-2.3 safetensors text
# encoder in slot 2 — that path caused validation-time loader crashes.
GEMMA_FILE = "gemma-3-12b-it-orthogonal-reflection-bounded-ablation-v4-12B-fp4_mixed.safetensors"
GEMMA_URL = ("https://huggingface.co/inflatebot/"
             "LTX23-gemma-3-12b-it-orthogonal-reflection-bounded-ablation-v4-fp4_mixed/"
             f"resolve/main/{GEMMA_FILE}")

display(HTML(f"<div style='padding:10px;background:#e3f2fd;border-left:4px solid #1976d2'>"
             f"<b>ComfyUI:</b> <code>{COMFY_PATH}</code><br>"
             f"<b>Workspace:</b> <code>{BASE_PATH}</code></div>"))

# ---- mediapipe pin (always needed regardless of template).
# 10S-Comfy-nodes' LTXFaceDetector calls mp.solutions, removed in
# mediapipe 0.10.14+. Without this pin face detection silently fails and
# LikenessGuide degrades to whole-frame reference -> source identity lost.
display(HTML("<p style='color:#00e676;font-weight:bold'>[1/4] Pinning mediapipe (10S-Comfy-nodes uses removed mp.solutions API)...</p>"))
run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "mediapipe==0.10.13"])

# ---- Custom node packs the Vantage workflow requires.
display(HTML("<p style='color:#00e676;font-weight:bold'>[2/4] Installing 11 custom node packs (idempotent — cached if present)...</p>"))
NODE_REPOS = [
    "https://github.com/kijai/ComfyUI-KJNodes",
    "https://github.com/city96/ComfyUI-GGUF",
    "https://github.com/Lightricks/ComfyUI-LTXVideo",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite",
    "https://github.com/kijai/ComfyUI-MelBandRoFormer",
    "https://github.com/TenStrip/10S-Comfy-nodes",
    "https://github.com/rgthree/rgthree-comfy",
    "https://github.com/pythongosssss/ComfyUI-Custom-Scripts",
    "https://github.com/ClownsharkBatwing/RES4LYF",
    "https://github.com/melMass/comfy_mtb",
    "https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes",
]
nodes_dir = f"{COMFY_PATH}/custom_nodes"
os.makedirs(nodes_dir, exist_ok=True)
for url in NODE_REPOS:
    name = url.rstrip("/").split("/")[-1]
    path = os.path.join(nodes_dir, name)
    if not os.path.exists(path):
        run(["git", "clone", "-q", url, path])
    req = f"{path}/requirements.txt"
    if os.path.exists(req):
        run([sys.executable, "-m", "pip", "install", "-q", "-r", req])
# Re-pin mediapipe AFTER custom-node requirements (some packs pull a newer
# mediapipe transitively and overwrite our 0.10.13).
run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "mediapipe==0.10.13"])

# ---- Try to use aria2c for fast multi-conn downloads.
if shutil.which("aria2c") is None:
    try:
        run(["apt-get", "update", "-qq"], check=False)
        run(["apt-get", "install", "-y", "-qq", "aria2"], check=False)
    except Exception: pass

def dl(url, dest, fname):
    import requests
    Path(dest).mkdir(parents=True, exist_ok=True)
    fpath = os.path.join(dest, fname)
    if os.path.exists(fpath) and os.path.getsize(fpath) > 0:
        print(f"  cached: {fname}"); return
    if shutil.which("aria2c"):
        run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16",
             "-k", "1M", "-d", dest, "-o", fname, url])
        return
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        tmp = fpath + ".part"
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk: f.write(chunk)
        os.replace(tmp, fpath)

# ---- 10Eros / Vantage model bundle (~24 GB total).
display(HTML("<p style='color:#00e676;font-weight:bold'>[3/4] Downloading Vantage / 10Eros models...</p>"))
B = COMFY_PATH + "/models"
MODELS = [
    # 10Eros UNET (Q4_K_M GGUF — best speed/quality tradeoff for L40s+).
    ("https://huggingface.co/vantagewithai/LTX2.3-10Eros-GGUF/resolve/main/10Eros_v1-Q4_K_M.gguf",
     "unet", "10Eros_v1-Q4_K_M.gguf"),
    # 10Eros's LTX-2.3 text encoder (slot 2 of DualCLIPLoader).
    ("https://huggingface.co/vantagewithai/LTX2.3-10Eros-Split/resolve/main/text_encoder/10Eros_v1_text_encoder.safetensors",
     "text_encoders", "10Eros_v1_text_encoder.safetensors"),
    # Gemma 3 12B (slot 1 of DualCLIPLoader) — single-file safetensors,
    # softly abliterated, drop-in for Vantage's expected file.
    (GEMMA_URL, "text_encoders", GEMMA_FILE),
    # 10Eros VAEs.
    ("https://huggingface.co/vantagewithai/LTX2.3-10Eros-Split/resolve/main/vae/10Eros_v1_vae.safetensors",
     "vae", "10Eros_v1_vae.safetensors"),
    ("https://huggingface.co/vantagewithai/LTX2.3-10Eros-Split/resolve/main/audio_vae/10Eros_v1_audio_vae.safetensors",
     "vae", "10Eros_v1_audio_vae.safetensors"),
    # OmniNFT LoRA (Vantage's canonical style LoRA; off by default in
    # photoreal mode via the Power Lora Loader toggle).
    ("https://huggingface.co/VasiliyWeb/OmniNFT_ComfyUI/resolve/main/OmniNFT_converted_lora.safetensors",
     "loras", "OmniNFT_converted_lora.safetensors"),
    # LTX-2.3 spatial upscaler v1.1 (stage-2 tiled sampler uses it).
    ("https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.1.safetensors",
     "latent_upscale_models", "ltx-2.3-spatial-upscaler-x2-1.1.safetensors"),
    # MelBand RoFormer (audio mel-spectrogram processor).
    ("https://huggingface.co/Kijai/MelBandRoFormer_comfy/resolve/main/MelBandRoformer_fp16.safetensors",
     "diffusion_models", "MelBandRoformer_fp16.safetensors"),
]
for url, sub, fname in MODELS:
    dl(url, f"{B}/{sub}", fname)

# Vantage workflow JSON (UI format — converted to API at submit time in Cell 3).
dl("https://huggingface.co/vantagewithai/LTX2.3-10Eros-Split/resolve/main/Vantage-10Eros_I2V_v3.2.json",
   str(BASE_PATH), "Vantage-10Eros_I2V_v3.2.json")

# Audio placeholder for the I2V workflow. Vantage's MathExpression node
# derives frame count from this file's duration; 1 second produces 17
# frames at fps=24. Generate via lavfi anullsrc.
audio_placeholder = os.path.join(COMFY_PATH, "input", "1.wav")
if not os.path.exists(audio_placeholder):
    os.makedirs(os.path.dirname(audio_placeholder), exist_ok=True)
    ffmpeg = shutil.which("ffmpeg") or "/usr/bin/ffmpeg"
    if os.path.exists(ffmpeg):
        run([ffmpeg, "-y", "-f", "lavfi", "-i", "anullsrc=r=44100:cl=mono",
             "-t", "1", "-c:a", "pcm_s16le", audio_placeholder], check=False)
    else:
        # Fallback: minimal 1-sec mono PCM WAV via Python.
        import wave, struct
        with wave.open(audio_placeholder, "wb") as w:
            w.setnchannels(1); w.setsampwidth(2); w.setframerate(44100)
            w.writeframes(struct.pack("<" + "h"*44100, *([0]*44100)))

# ---- Sanity check.
display(HTML("<p style='color:#00e676;font-weight:bold'>[4/4] Verifying mediapipe.solutions + Vantage workflow + Gemma file...</p>"))
import importlib, mediapipe as mp
importlib.reload(mp)
assert hasattr(mp, "solutions") and hasattr(mp.solutions, "face_detection"), \
    f"mediapipe {mp.__version__} missing mp.solutions — re-run this cell"
assert os.path.exists(WF_UI_PATH), f"Workflow file missing: {WF_UI_PATH}"
assert os.path.exists(f"{B}/text_encoders/{GEMMA_FILE}"), f"Gemma file missing: {GEMMA_FILE}"
assert os.path.exists(f"{B}/unet/10Eros_v1-Q4_K_M.gguf"), "10Eros UNET missing"
assert os.path.exists(f"{B}/loras/OmniNFT_converted_lora.safetensors"), "OmniNFT LoRA missing"
display(HTML(f"<div style='padding:12px;background:#e8f5e9;border-left:4px solid #4caf50;color:#2e7d32'>"
             f"<b>Setup complete.</b><br>"
             f"mediapipe <code>{mp.__version__}</code> — mp.solutions OK<br>"
             f"ComfyUI: <code>{COMFY_PATH}</code><br>"
             f"Vantage workflow: <code>{WF_UI_PATH}</code><br>"
             f"Gemma: <code>{GEMMA_FILE}</code> (safetensors, softly abliterated)<br>"
             f"Audio placeholder: <code>1.wav</code> (1 sec → 17 frames via Vantage MathExpression)"
             f"</div>"))

In [ ]:
# Cell 2/3 — UI->API converter (inline port of tools/ui_to_api.py).
#
# Vantage workflow JSON is in UI format (the ComfyUI canvas save format).
# ComfyUI's /prompt endpoint requires API format. The converter walks the
# UI nodes, queries /object_info for the live node-class schema, and
# emits {nid: {class_type, inputs}}. Power Lora Loader (rgthree) needs a
# special case because its widget values are dict-shaped
# {on, lora, strength, strengthTwo} and the generic mapper drops them.
import json, os, requests
from typing import Any

COMFY_URL = "http://127.0.0.1:8188"
WIDGET_TYPE_NAMES = {"COMBO","INT","FLOAT","STRING","BOOLEAN",
                     "COMFY_DYNAMICCOMBO_V3","COMFY_DYNAMICCOMBO",
                     "COMFY_MULTILINESTRING_V3","COMFY_MULTILINESTRING"}

def _is_widget(t):
    if isinstance(t, list) and t:
        f = t[0]
        if isinstance(f, list): return True
        if isinstance(f, str) and f in WIDGET_TYPE_NAMES: return True
    return False

def _map_widgets(info, vals):
    inp = info.get("input", {}) or {}
    order = info.get("input_order", {}) or {}
    if isinstance(vals, dict): return dict(vals)
    if not isinstance(vals, list): return {}
    out, idx = {}, 0
    for sec in ("required", "optional"):
        keys = order.get(sec) or list((inp.get(sec) or {}).keys())
        for k in keys:
            v = (inp.get(sec) or {}).get(k)
            if not _is_widget(v): continue
            if idx >= len(vals): return out
            out[k] = vals[idx]
            idx += 1
            if isinstance(v, list) and v[0] in ("COMFY_DYNAMICCOMBO_V3","COMFY_DYNAMICCOMBO"):
                opts = v[1].get("options", []) if len(v) >= 2 and isinstance(v[1], dict) else []
                for opt in opts:
                    if isinstance(opt, dict) and opt.get("key") == vals[idx-1]:
                        for nk, nv in (opt.get("inputs") or {}).get("required", {}).items():
                            if _is_widget(nv) and idx < len(vals):
                                out[f"{k}.{nk}"] = vals[idx]; idx += 1
                        break
    return out

def ui_to_api(ui_wf, object_info):
    nodes = ui_wf.get("nodes") or []
    links = ui_wf.get("links") or []
    skipped = {str(n.get("id")) for n in nodes if n.get("mode") in (2, 4)}
    link_src = {}
    for raw in links:
        if not isinstance(raw, list) or len(raw) < 5: continue
        link_src[int(raw[0])] = (str(raw[1]), int(raw[2]))
    api = {}
    for n in nodes:
        nid = str(n.get("id")); ctype = n.get("type")
        if not ctype or ctype in ("MarkdownNote","Note","PrimitiveNode"): continue
        if n.get("mode") in (2, 4): continue
        info = object_info.get(ctype)
        if info is None:
            stub = {}
            for s in (n.get("inputs") or []):
                if s.get("link") is None: continue
                src = link_src.get(int(s["link"]))
                if src: stub[s.get("name")] = [src[0], src[1]]; break
            api[nid] = {"class_type": ctype, "inputs": stub}
            print(f"  [warn] unknown {ctype!r} (node {nid})")
            continue
        wv = n.get("widgets_values") or []
        inputs = _map_widgets(info, wv)
        # Power Lora Loader special case. Vantage stores LoRA paths with a
        # backslash subdir prefix like 'ltx23\OmniNFT_converted_lora.safetensors'.
        # That's ONE literal backslash in the runtime string. The check must
        # be "\\" in Python source (= one literal backslash at runtime).
        # NOT "\\\\" (= two literal backslashes — would never match and the
        # OmniNFT file would resolve as 'ltx23\\OmniNFT_converted_lora...'
        # which doesn't exist on disk, so the LoRA fails to load silently).
        if ctype == "Power Lora Loader (rgthree)" and isinstance(wv, list):
            slot = 0
            for x in wv:
                if isinstance(x, dict) and "lora" in x and "strength" in x:
                    lp = x.get("lora") or ""
                    if "\\" in lp: lp = lp.rsplit("\\", 1)[-1]
                    if "/" in lp: lp = lp.rsplit("/", 1)[-1]
                    slot += 1
                    inputs[f"lora_{slot}"] = {"on": bool(x.get("on", True)),
                                              "lora": lp,
                                              "strength": float(x.get("strength", 1.0)),
                                              "strengthTwo": x.get("strengthTwo")}
        for s in (n.get("inputs") or []):
            if s.get("link") is None: continue
            src = link_src.get(int(s["link"]))
            if src and src[0] not in skipped:
                inputs[s.get("name")] = [src[0], src[1]]
        api[nid] = {"class_type": ctype, "inputs": inputs}
    # Bypass-pass: drop unknown nodes whose first input shape is passthrough
    unknown = [nid for nid, n in api.items() if n["class_type"] not in object_info]
    pt = {}
    for nid in unknown:
        for v in api[nid]["inputs"].values():
            if isinstance(v, list) and len(v) >= 2: pt[nid] = v; break
    for n in api.values():
        for k, v in list(n["inputs"].items()):
            if isinstance(v, list) and len(v) >= 1 and str(v[0]) in pt:
                n["inputs"][k] = pt[str(v[0])]
    for nid in unknown:
        if nid in pt: api.pop(nid, None)
    return api

# Self-test: make sure the backslash fix is in place. If you see this print,
# the converter is loaded and the LoRA prefix strip will work correctly.
assert "\\\\" not in __import__("inspect").getsource(ui_to_api), \
    "Backslash escape bug still present — re-run cell with fixed source"
print("converter loaded — Vantage LoRA path prefix strip OK (1 backslash)")

In [ ]:
# Cell 3/3 — Boot ComfyUI + Gradio UI with dry-run/smoke/full modes
#
# Fast-iteration modes (key feature — saves you from $$ on 10-min trials):
#   dry-run : validate prompt against ComfyUI's node schema, NO sampling.
#             Catches strength-out-of-range / missing-file / class-not-found
#             errors in ~30 sec. ALWAYS run this first after a config change.
#   smoke   : 2 stage-1 + 1 stage-2 steps at 384x512. ~1-2 min on L40s+.
#             Use to verify the workflow runs end-to-end before paying for
#             full quality.
#   full    : Vantage default 13+3 steps at requested resolution. ~3-5 min
#             on L40s, ~2-3 min on H100.
#
# Face anchoring philosophy (per TenStrip's own YouTube demo of ltx23face.json):
#   - LikenessAnchor / LatentAnchorAware are STABILIZATION layers, NOT
#     correction layers. Recommended strength is 0.11, NOT the schema max.
#     "The lower the strength, the more natural the motion feels. You're
#     keeping the identity from drifting, not locking the face completely."
#   - Longer clips (97+ frames) let the anchor stabilize better than short
#     clips (17 frames). 17 frames is the workflow minimum from audio sync.
#   - Even with correct settings, LTX-2.3 has inherent face drift. For
#     pixel-perfect identity preservation, Wan 2.2 is the community-
#     recommended model. LTX-2.3 + 10Eros + 10S nodes preserves SUBJECT
#     CATEGORY (Indian woman, brown skin, etc.) and FRAME-TO-FRAME
#     CONSISTENCY but not strict source-photo identity.
import os, sys, subprocess, shutil, socket, time, glob, json, threading
import urllib.request, urllib.error
from pathlib import Path
from PIL import Image
import requests
import gradio as gr
from IPython.display import display, HTML

# ---- Re-discover paths if Cell 1 didn't run in this kernel session.
if "COMFY_PATH" not in globals() or not os.path.exists(os.path.join(globals().get("COMFY_PATH",""), "main.py")):
    CANDIDATES = [
        "/teamspace/studios/this_studio/ComfyUI",
        "/teamspace/studios/this_studio/comfyui",
        "/workspace/ComfyUI",
        "/root/ComfyUI",
        str(Path.home() / "ComfyUI"),
    ]
    COMFY_PATH = next((c for c in CANDIDATES if os.path.exists(os.path.join(c, "main.py"))), None)
    if COMFY_PATH is None:
        raise RuntimeError(
            "ComfyUI not found. Run Cell 1 first, OR ensure your Lightning AI "
            "studio was launched from the Clean ComfyUI Template.")
    BASE_PATH = Path(COMFY_PATH).parent
    WF_UI_PATH = str(BASE_PATH / "Vantage-10Eros_I2V_v3.2.json")

COMFY_URL = "http://127.0.0.1:8188"
COMFY_LOG_PATH = str(BASE_PATH / "comfyui_server.log")
OUTPUT_PATH = f"{COMFY_PATH}/output"
INPUT_PATH  = f"{COMFY_PATH}/input"
GEMMA_FILE = globals().get("GEMMA_FILE",
    "gemma-3-12b-it-orthogonal-reflection-bounded-ablation-v4-12B-fp4_mixed.safetensors")
LTX_TEXT_ENCODER_FILE = "10Eros_v1_text_encoder.safetensors"

# ---- ComfyUI server lifecycle.
def is_server_running(port=8188):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

def boot_server():
    env = os.environ.copy()
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    log = open(COMFY_LOG_PATH, "a", encoding="utf-8")
    proc = subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
                            cwd=COMFY_PATH, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    def stream():
        for line in proc.stdout:
            print(line, end="", flush=True); log.write(line); log.flush()
    threading.Thread(target=stream, daemon=True).start()
    t0 = time.time()
    while not is_server_running():
        if time.time() - t0 > 300:
            raise RuntimeError(f"ComfyUI failed to start. See {COMFY_LOG_PATH}")
        time.sleep(2)

def load_and_convert_workflow():
    ui = json.loads(Path(WF_UI_PATH).read_text(encoding="utf-8"))
    info = requests.get(f"{COMFY_URL}/object_info", timeout=60).json()
    return ui_to_api(ui, info)   # ui_to_api defined in Cell 2

def queue_prompt(wf):
    r = requests.post(f"{COMFY_URL}/prompt", json={"prompt": wf}, timeout=30)
    if r.status_code >= 400:
        raise RuntimeError(f"/prompt {r.status_code}: {r.text[:500]}")
    return r.json()

def get_latest_video():
    mp4s = (glob.glob(f"{OUTPUT_PATH}/**/*.mp4", recursive=True)
            + glob.glob(f"{OUTPUT_PATH}/*.mp4"))
    return max(mp4s, key=os.path.getctime) if mp4s else None

# Vantage's default cfg_values is "2,1.5,1,1,1,1,1,1,1,1,1,1,1" — after
# step 2 the CFG drops to 1.0 which mathematically nullifies the negative
# prompt for 11 of 13 steps. For photoreal output we extend cfg > 1 across
# the early steps (all values within STGGuiderAdvanced's accepted range).
PHOTOREAL_CFG = "3,2.5,2,2,1.8,1.5,1.5,1.3,1.2,1.2,1.1,1,1"
PHOTOREAL_STG = "3,2.5,2,2,1.8,1.5,1.5,1.3,1.2,1.2,1.1,1,1"
SMOKE_SIGMAS = "1.0,0.5,0.0"
SMOKE_CFG    = "2,1.5"
SMOKE_STG    = "2,1.5"

def patch_workflow_for_run(api_wf, mode, image_filepath, prompt, neg_prompt,
                           width, height, seed, omninft_strength,
                           photoreal_cfg=True, smoke=False):
    """Stamp user inputs onto the Vantage API workflow. All strength values
    stay at Vantage's defaults (NOT bumped to schema max) — bumping anchor
    strengths goes AGAINST the TenStrip designer's intent per their YouTube
    demo of ltx23face.json. Anchor nodes are stabilization, not correction."""
    if mode == "Image-to-Video":
        if not image_filepath:
            raise gr.Error("Please upload an image for I2V.")
        os.makedirs(INPUT_PATH, exist_ok=True)
        fname = os.path.basename(image_filepath)
        shutil.copy(image_filepath, os.path.join(INPUT_PATH, fname))
        for n in api_wf.values():
            if n.get("class_type") == "LoadImage": n["inputs"]["image"] = fname
    else:
        os.makedirs(INPUT_PATH, exist_ok=True)
        Image.new("RGB", (width, height), "black").save(
            os.path.join(INPUT_PATH, "_t2v_blank.png"))
        for n in api_wf.values():
            if n.get("class_type") == "LoadImage": n["inputs"]["image"] = "_t2v_blank.png"

    # Ensure DualCLIPLoader points at our installed Gemma + LTX text encoder.
    for n in api_wf.values():
        if n.get("class_type") == "DualCLIPLoader":
            ins = n.setdefault("inputs", {})
            ins["clip_name1"] = GEMMA_FILE
            ins["clip_name2"] = LTX_TEXT_ENCODER_FILE
        if n.get("class_type") == "CLIPLoader":
            n.setdefault("inputs", {})["clip_name"] = GEMMA_FILE

    # Dimensions — patch the INTConstants feeding ImageResizeKJv2.
    for rnid, rn in api_wf.items():
        if rn.get("class_type") not in ("ImageResizeKJv2", "ImageResizeKJ"): continue
        for role, target in (("width", int(width)), ("height", int(height))):
            ref = rn["inputs"].get(role)
            if isinstance(ref, list) and len(ref) >= 1:
                src = api_wf.get(str(ref[0]))
                if src and src.get("class_type") in ("INTConstant", "PrimitiveInt"):
                    src.setdefault("inputs", {})["value"] = target
    for n in api_wf.values():
        if n.get("class_type") == "EmptyLTXVLatentVideo":
            n.setdefault("inputs", {})["width"]  = int(width)
            n["inputs"]["height"] = int(height)

    # Trace guider -> CLIPTextEncode for prompts.
    def trace_clip_text(start, slot, hops=8):
        seen=set(); cur=api_wf.get(start,{}).get("inputs",{}).get(slot)
        while isinstance(cur,list) and cur and hops>0:
            nid=str(cur[0])
            if nid in seen: break
            seen.add(nid)
            n=api_wf.get(nid,{})
            if n.get("class_type")=="CLIPTextEncode": return nid
            cur=next((v for v in n.get("inputs",{}).values() if isinstance(v,list)),None)
            hops-=1
        return None
    guider = next((nid for nid,n in api_wf.items()
                   if n.get("class_type") in ("STGGuiderAdvanced","CFGGuider","STGGuider")), None)
    if guider:
        p = trace_clip_text(guider, "positive"); ng = trace_clip_text(guider, "negative")
        if p:  api_wf[p]["inputs"]["text"]  = prompt
        if ng: api_wf[ng]["inputs"]["text"] = neg_prompt

    # CFG + sigmas schedule — smoke shortens, photoreal extends the CFG>1 window.
    for n in api_wf.values():
        if n.get("class_type") == "STGGuiderAdvanced":
            ins = n.setdefault("inputs", {})
            if smoke:
                ins["sigmas"] = SMOKE_SIGMAS
                ins["cfg_values"] = SMOKE_CFG
                ins["stg_scale_values"] = SMOKE_STG
                ins["stg_rescale_values"] = "1,1"
                ins["stg_layers_indices"] = "[9999],[9999]"
            elif photoreal_cfg:
                ins["cfg_values"] = PHOTOREAL_CFG
                ins["stg_scale_values"] = PHOTOREAL_STG

    # IMPORTANT: do NOT bump LikenessGuide/Anchor strengths. Vantage's
    # defaults (LikenessGuide=0.9, LikenessAnchor=0.5) and TenStrip's
    # ltx23face.json (LatentAnchorAware=0.11) are intentionally LOW.
    # Bumping to schema max causes the model to lock onto the source's
    # "category" (Indian woman) instead of the source's "identity"
    # (this specific person). Leave the workflow defaults intact.

    # OmniNFT strength via Vantage's own Power Lora Loader toggle.
    for n in api_wf.values():
        if n.get("class_type") == "Power Lora Loader (rgthree)":
            for v in n.get("inputs", {}).values():
                if isinstance(v, dict) and "OmniNFT" in str(v.get("lora","")):
                    v["strength"] = float(omninft_strength)
                    v["on"] = float(omninft_strength) > 0

    # Seeds.
    for n in api_wf.values():
        ins = n.get("inputs", {})
        if "noise_seed" in ins: ins["noise_seed"] = int(seed)
        if "seed" in ins and isinstance(ins["seed"], (int, float)): ins["seed"] = int(seed)
    return api_wf

def dry_run(mode, image_filepath, prompt, neg_prompt, width, height, seed,
            omninft_strength, photoreal_cfg):
    """Validate without sampling. Catches schema errors in ~30 sec."""
    if not is_server_running(): boot_server()
    W = max(512, round(width / 32) * 32)
    H = max(512, round(height / 32) * 32)
    api = load_and_convert_workflow()
    api = patch_workflow_for_run(api, mode, image_filepath, prompt, neg_prompt,
                                 W, H, seed, omninft_strength, photoreal_cfg, smoke=False)
    r = requests.post(f"{COMFY_URL}/prompt",
                      json={"prompt": api, "extra_data": {"client_id": "dryrun"}},
                      timeout=30)
    if r.status_code >= 400:
        return f"VALIDATION FAILED:\n{r.text[:800]}"
    pid = r.json().get("prompt_id")
    time.sleep(1)
    try:
        h = requests.get(f"{COMFY_URL}/history/{pid}", timeout=10).json()
        if pid in h:
            st = h[pid].get("status", {})
            if st.get("status_str") == "error":
                return f"VALIDATION FAILED:\n{json.dumps(st.get('messages',[])[:5], indent=2)}"
    except Exception: pass
    requests.post(f"{COMFY_URL}/interrupt", timeout=5)
    return "VALIDATION OK — submit smoke or full run next."

def generate_video(run_mode, mode, image_filepath, prompt, neg_prompt, width,
                   height, seed, omninft_strength, photoreal_cfg,
                   progress=gr.Progress()):
    if run_mode == "dry-run":
        progress(0.5, desc="Validating workflow against ComfyUI schema...")
        result = dry_run(mode, image_filepath, prompt, neg_prompt, width, height,
                         seed, omninft_strength, photoreal_cfg)
        progress(1.0, desc="Done")
        raise gr.Error(result if "FAILED" in result else f"DRY-RUN: {result}")

    smoke = (run_mode == "smoke")
    eff_w, eff_h = (384, 512) if smoke else (width, height)
    if not is_server_running():
        progress(0.05, desc="Booting ComfyUI...")
        boot_server()
    W = max(512, round(eff_w / 32) * 32)
    H = max(512, round(eff_h / 32) * 32)
    progress(0.1, desc="Converting Vantage workflow UI -> API...")
    api = load_and_convert_workflow()
    progress(0.2, desc=f"Patching workflow ({run_mode} mode)...")
    api = patch_workflow_for_run(api, mode, image_filepath, prompt, neg_prompt,
                                 W, H, seed, omninft_strength, photoreal_cfg, smoke=smoke)
    progress(0.3, desc="Queuing /prompt...")
    pid = queue_prompt(api)["prompt_id"]
    eta_min = "1-2" if smoke else "3-6"
    progress(0.4, desc=f"Sampling (~{eta_min} min on L40s+)...")
    while True:
        try:
            h = json.loads(urllib.request.urlopen(
                f"{COMFY_URL}/history/{pid}", timeout=10).read())
            if str(pid) in h:
                st = h[str(pid)].get("status", {})
                if st.get("status_str") == "error":
                    raise gr.Error(f"ComfyUI rejected: {st.get('messages',[])[:3]}")
                break
            q = json.loads(urllib.request.urlopen(
                f"{COMFY_URL}/queue", timeout=10).read())
            if not any(str(j[1]) == str(pid)
                       for j in q.get("queue_running",[])+q.get("queue_pending",[])):
                raise gr.Error("Generation crashed — see ComfyUI log")
        except urllib.error.URLError: pass
        time.sleep(3)
    progress(1.0, desc="Done")
    return get_latest_video()

# ---- Gradio UI. Use Lightning's Custom Port plugin to forward 7860.
with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("# 10Eros LTX-2.3 — Vantage Workflow on Lightning AI")
    gr.Markdown("""Vantage I2V_v3.2 + 10Eros UNET + inflatebot's softly-abliterated Gemma 3 12B. Uses TenStrip's own anchor-strength philosophy: low strengths for natural motion, not face-locking.

**Iteration modes (top-right):**
- **dry-run** (~30 s): validate config against ComfyUI schema, NO sampling. Run after ANY change.
- **smoke** (~1-2 min): 2 stage-1 steps at 384×512. End-to-end check before quality.
- **full** (~3-6 min): Vantage default 13+3 steps at requested resolution.

**Honest about face identity:** LTX-2.3 preserves subject *category* + frame-to-frame consistency well. For *exact* source-photo identity, train a character LoRA (5-min training on 8-15 photos via `ostris/ai-toolkit`) or use **Wan 2.2** which the LTX-2.3 community itself recommends for stricter face consistency.""")
    with gr.Row():
        with gr.Column(scale=1):
            run_mode_sel = gr.Radio(["dry-run", "smoke", "full"], value="dry-run",
                                    label="Run mode (always dry-run first after a change)")
            mode_sel = gr.Radio(["Image-to-Video", "Text-to-Video"],
                                value="Image-to-Video", label="Mode")
            img_in = gr.Image(type="filepath", label="Source portrait")
            prompt = gr.Textbox(label="Prompt",
                                value="cinematic close-up portrait, soft gentle smile forming, subtle breathing motion, photorealistic, sharp focus, natural skin texture, studio lighting",
                                lines=3)
            neg = gr.Textbox(label="Negative prompt",
                             value="anime, cartoon, drawing, illustration, painting, 3d render, cgi, stylized, blurry, distorted, deformed",
                             lines=2)
            with gr.Row():
                w = gr.Slider(512, 1344, step=32, value=768, label="Width (full mode)")
                hh = gr.Slider(512, 1344, step=32, value=1024, label="Height (full mode)")
            with gr.Row():
                seed_in = gr.Number(value=42, label="Seed", precision=0)
                omni = gr.Slider(0.0, 1.5, step=0.05, value=0.0,
                                 label="OmniNFT (0=photoreal, 0.8=Vantage default anime)")
            photoreal_chk = gr.Checkbox(value=True,
                label="Extend CFG schedule (keeps negative prompt active across all sampler steps)")
            gen_btn = gr.Button("Run", variant="primary")
        with gr.Column(scale=1):
            vid_out = gr.Video(label="Result")
    mode_sel.change(lambda m: gr.update(visible=(m == "Image-to-Video")),
                    inputs=mode_sel, outputs=img_in)
    gen_btn.click(generate_video,
                  inputs=[run_mode_sel, mode_sel, img_in, prompt, neg, w, hh, seed_in, omni, photoreal_chk],
                  outputs=vid_out)
demo.launch(share=True, inline=True, server_name="0.0.0.0", server_port=7860)